# Batch extraction with `ai_query()`

Uses `ai_query()` for batch extraction ("`ai_query()` for batch —
not a Python loop").

## What this does

Runs the ClaimClerk extraction prompt across all ~500 customer
emails in `/Volumes/${catalog}/pawshield/bootstrap/claim_emails/`
using `ai_query()` in **one SQL statement**. The SQL warehouse
parallelises the calls across workers; the result lands in
`${catalog}.pawshield.email_extracts` as a typed STRUCT column.

The one-SQL-statement form beats a Python-loop alternative.
This notebook lets you measure the difference on your workspace.

## What you'll have at the end

* `${catalog}.pawshield.email_extracts` Delta table with one row
  per `.eml` file, containing source_path + extracted STRUCT +
  processed_at timestamp.
* A verification query that pulls Sarah Chen's row
  (`sarah_chen_20260328_098473.eml`) and shows the extracted
  `claim_id`, `intent`, `urgency`, etc.

## Prerequisites

* The setup notebook c0001 has run (the `bootstrap` Volume is populated
  with the 500 `.eml` files).
* The endpoint `databricks-meta-llama-3-1-8b-instruct` is available
  on your workspace (Foundation Model API).
* Serverless compute (Databricks default) — runtime 15.4 LTS+ for
  `responseFormat`, or 17.3+ for the broadest argument support.

## Source verification

The `ai_query()` signature used here was verified May 2026 against
the [Databricks SQL function reference](https://docs.databricks.com/aws/en/sql/language-manual/functions/ai_query).
`responseFormat` is the documented structured-output argument for
chat foundation models; `returnType` is retained for non-chat
endpoints where the response shape isn't schema-constrained at
the model.

In [0]:
# Pin mlflow<3.13 — 3.13.0 has a regression where eval_item.trace is None
# during mlflow.genai.evaluate (trace not associated with eval_request_id).
%pip install --quiet --force-reinstall "mlflow[databricks]>=3.12,<3.13"

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
jupyter-server 1.23.4 requires anyio<4,>=3.1.0, but you have anyio 4.13.0 which is incompatible.
databricks-connect 14.3.18 requires numpy<2,>=1.15, but you have numpy 2.2.6 which is incompatible.
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
dbutils.library.restartPython()

In [0]:
dbutils.widgets.text("catalog", "genaicert")
dbutils.widgets.text("llm_endpoint", "databricks-meta-llama-3-1-8b-instruct")
dbutils.widgets.text("prompt_name", "claimclerk_extraction")
dbutils.widgets.text("source_glob", "claim_emails/*.eml")
catalog = dbutils.widgets.get("catalog")
llm_endpoint = dbutils.widgets.get("llm_endpoint")
prompt_short_name = dbutils.widgets.get("prompt_name")
source_glob = dbutils.widgets.get("source_glob")

In [0]:
print(f"Catalog:      {catalog}")
print(f"LLM endpoint: {llm_endpoint}")
print(f"Prompt:       {catalog}.pawshield.{prompt_short_name}")
print(f"Source glob:  /Volumes/{catalog}/pawshield/bootstrap/{source_glob}")

Catalog:      genaicert
LLM endpoint: databricks-meta-llama-3-1-8b-instruct
Prompt:       genaicert.pawshield.claimclerk_extraction
Source glob:  /Volumes/genaicert/pawshield/bootstrap/claim_emails/*.eml


## 1. Pull the extraction prompt from MLflow Prompt Registry

The c0301 notebook registers `claimclerk_extraction` and sets the
`@champion` alias. We load it here so the batch job uses the same
prompt as the live chain.

In [0]:
import mlflow

PROMPT_NAME = f"{catalog}.pawshield.{prompt_short_name}"

# Fallback: inline the four-part prompt for readers who haven't run the
# c0301 notebook yet. Keep this in sync with that notebook's registered prompt.
FALLBACK_SYSTEM_PROMPT = (
    "ROLE: You extract structured fields from PawShield customer "
    "claim emails.\n\n"
    "TASK: Given the raw email text, return ONLY a JSON object "
    "with keys: claim_id (string or null), customer_id (string or "
    "null), pet_name (string or null), contact_phone (string or "
    "null), intent (one of: claim_status, policy_q, appeal, "
    "complaint, vet_lookup, emergency, other), sentiment (one of: "
    "positive, neutral, frustrated, angry, panicked), urgency (one "
    "of: low, medium, high, emergency).\n\n"
    "CONSTRAINTS: Output JSON only — no commentary, no preamble. "
    "Use null when a field isn't present in the email. Pet names "
    "are first-letter-capitalised. Phone numbers in hyphenated "
    "format like 512-555-0188.\n\n"
    "EXAMPLE: Input email mentioning 'Biscuit's ER claim "
    "CLM-2026-03-00471 from Sarah Chen at 512-555-0188, appealing "
    "the $512 reimbursement' produces "
    '{"claim_id":"CLM-2026-03-00471","customer_id":null,'
    '"pet_name":"Biscuit","contact_phone":"512-555-0188",'
    '"intent":"appeal","sentiment":"frustrated","urgency":"medium"}.'
)

In [0]:
try:
    prompt = mlflow.genai.load_prompt(f"prompts:/{PROMPT_NAME}@champion")
    system_prompt = prompt.template
    print(f"Loaded {PROMPT_NAME}@champion (version {prompt.version}).")
except Exception as e:
    print(
        f"Prompt Registry lookup failed ({type(e).__name__}: {e}); "
        "falling back to the inline four-part prompt. Run the c0301 "
        "notebook to populate the registry."
    )
    system_prompt = FALLBACK_SYSTEM_PROMPT

Loaded genaicert.pawshield.claimclerk_extraction@champion (version 11).


## 2. Create the output table

`email_extracts` lands one row per source `.eml`. The
`extracted` column is a typed STRUCT — `ai_query` with a
JSON-Schema `responseFormat` returns a JSON STRING, which the
INSERT below parses with `from_json` against the same schema.

In [0]:
%sql
CREATE TABLE IF NOT EXISTS IDENTIFIER(:catalog || '.pawshield.email_extracts') (
    source_path   STRING,
    extracted     STRUCT<
                      claim_id:STRING,
                      customer_id:STRING,
                      pet_name:STRING,
                      contact_phone:STRING,
                      intent:STRING,
                      sentiment:STRING,
                      urgency:STRING
                    >,
    processed_at  TIMESTAMP
) USING DELTA;

## 3. The one-statement batch

`read_files(..., format => 'text')` reads each `.eml` as a row
with `path` and `value` columns. `ai_query` is called per row;
the warehouse parallelises the calls. `responseFormat` (JSON
Schema form) constrains the model's output at decoding time;
`from_json` then parses the returned JSON STRING into the
table's typed STRUCT column in a single SQL expression.

**`wholetext => true` matters.** Without this option, Spark's
text reader returns *one row per line of every file* — a 17-line
email turns into 17 source rows, all sharing the same
`_metadata.file_path`, and the MERGE inserts 17 rows for that
file on first run (the per-line value isn't a complete email so
the LLM extraction is partial and useless). The Apache Spark
text-files doc defines `wholetext`: *"If true, read each file
from input path(s) as a single row."* — the right shape for
whole-email-per-row LLM extraction. If you already ran this
notebook without `wholetext`, drop the table once before
re-running so the multi-line stale rows don't poison the
verification step (§4 below): `DROP TABLE
genaicert.pawshield.email_extracts`.

**Idempotent by construction.** The batch is written as a `MERGE`
that (a) filters out files whose extraction already succeeded and
(b) UPDATEs failed rows / INSERTs new rows in one pass. Re-running
the cell is safe and cheap — only files that are new or
previously-failed cost LLM tokens. This collapses what used to be
a separate "retry the failed rows" cell into the same statement
the batch runs; resilience belongs in the production path, not as
an afterthought cell readers might forget to run.

The `failOnError => false` argument changes the `ai_query` return
shape from a bare JSON STRING to a STRUCT with `.result` (the
successful JSON STRING) and `.errorMessage` (populated on failure
rows; NULL on success). The MERGE extracts `.result` and parses
it with `from_json`; failed rows land with `extracted = NULL` and
the next re-run picks them up via the WHERE filter.
Source: https://docs.databricks.com/aws/en/sql/language-manual/functions/ai_query
(Runtime field names: `.result` + `.errorMessage`. The published
doc still describes these as `.response` + `.errorStatus` — doc
lag; the runtime emits `result`/`errorMessage`. The book teaches
the runtime form; on the exam, recognise either as the same fields.)

**Note for production scale** (>few-thousand rows or evolving
extraction schema): the fused `ai_query` + `from_json` shape
below is right for tutorial clarity but couples an expensive
non-deterministic step (the LLM call) to a cheap deterministic
step (the JSON parse). A team adding a new extraction field
later has to re-run `ai_query` for every historical row to
repopulate the table — paying LLM tokens for what should be a
pure-SQL re-parse. The bronze/silver split ("Production consideration: split
bronze (raw LLM output) from silver (parsed STRUCT)" describes
the two-table pattern that makes schema evolution cheap; reach
for it when batch volume and schema-change frequency justify the
extra table.

In [0]:
# Use a Python f-string to inject the system prompt + catalog into SQL.
# (IDENTIFIER + parameter binding handles the catalog name safely; the
# prompt body is a STRING literal we embed in the request expression.)
escaped_prompt = system_prompt.replace("'", "''")  # single-quote escape for SQL

# Source: https://docs.databricks.com/aws/en/sql/language-manual/functions/ai_query
#   - endpoint        : STRING literal naming the serving endpoint
#   - request         : expression evaluated to STRING per row
#   - responseFormat  : JSON Schema envelope; constrains model output at decode time
#   - failOnError     : false => returns STRUCT<result:STRING, errorMessage:STRING>
#                       Success rows have `result` populated + `errorMessage=NULL`;
#                       failure rows have `result=NULL` + `errorMessage` populated.
#                       The job completes regardless of per-row failures instead of
#                       aborting on the first. Field names are the runtime form
#                       (the published doc still describes these as
#                       `response`/`errorStatus` — doc-vs-runtime lag; the runtime
#                       emits `result`/`errorMessage`). The book teaches the runtime
#                       form throughout.
#                       We extract .result (a JSON STRING) and parse with from_json.
#
# Idempotent: uses MERGE + filters out already-successful extractions so
# re-runs only spend LLM tokens on new or previously-failed files.
# The system prompt (loaded from the registry @champion in §1) lists
# the intent/sentiment/urgency vocabularies and the pet_name shape, but
# prose constraints alone don't bind the model — it can pack
# chain-of-thought into a free-text field like pet_name or drop a
# categorical field such as intent. The fix is to
# enforce the same vocabularies at the structured-output decoder level:
# `enum` on the categorical fields, `maxLength` on the prose-prone
# free-text fields. With `strict: true` the decoder rejects any token
# stream that violates the schema, so the model is forced to commit to
# a real value rather than fall back to null or to free-text prose.
#
# Pedagogy: prompt + schema together — either alone is fragile. The
# system prompt teaches the model what the values *mean*; the schema's
# enum + maxLength enforce them at decode time. Treat the schema as a
# load-bearing safety rail, not a type hint.
EXTRACT_JSON_SCHEMA = '''{
    "type": "json_schema",
    "json_schema": {
        "name": "claim_extraction",
        "schema": {
            "type": "object",
            "properties": {
                "claim_id":      {"type": ["string", "null"], "maxLength": 25},
                "customer_id":   {"type": ["string", "null"], "maxLength": 40},
                "pet_name":      {"type": ["string", "null"], "maxLength": 40},
                "contact_phone": {"type": ["string", "null"], "maxLength": 20},
                "intent":        {"type": "string", "enum": ["claim_status", "policy_q", "appeal", "complaint", "vet_lookup", "emergency", "other"]},
                "sentiment":     {"type": "string", "enum": ["positive", "neutral", "frustrated", "angry", "panicked"]},
                "urgency":       {"type": "string", "enum": ["low", "medium", "high", "emergency"]}
            },
            "required": ["claim_id", "customer_id", "pet_name", "contact_phone", "intent", "sentiment", "urgency"]
        },
        "strict": true
    }
}'''
EXTRACT_STRUCT_DDL = (
    "STRUCT<claim_id:STRING, customer_id:STRING, pet_name:STRING, "
    "contact_phone:STRING, intent:STRING, sentiment:STRING, urgency:STRING>"
)
batch_sql = f"""
MERGE INTO `{catalog}`.pawshield.email_extracts AS t
USING (
    SELECT
        _metadata.file_path AS source_path,
        from_json(
            ai_query(
                endpoint       => '{llm_endpoint}',
                request        => CONCAT('{escaped_prompt}\\n\\nEMAIL:\\n', value),
                responseFormat => '{EXTRACT_JSON_SCHEMA}',
                failOnError    => false
            ).result,
            '{EXTRACT_STRUCT_DDL}'
        ) AS extracted,
        current_timestamp() AS processed_at
    FROM read_files(
        '/Volumes/{catalog}/pawshield/bootstrap/{source_glob}',
        format => 'text',
        wholetext => 'true'
    )
    WHERE _metadata.file_path NOT IN (
        SELECT source_path
        FROM `{catalog}`.pawshield.email_extracts
        WHERE extracted IS NOT NULL
    )
) AS s
ON t.source_path = s.source_path
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *
"""

In [0]:
spark.sql(batch_sql)
print("Batch complete (MERGE — idempotent).")

Batch complete (MERGE — idempotent).


In [0]:
# Catches the wholetext-missing failure mode early: if total rows exceed
# the number of source .eml files, the source CTE is producing multiple
# rows per file (Spark text reader returned one row per line). Drop the
# table and re-run after confirming `wholetext => true` is on the
# read_files call above.
file_count = (
    spark.read.format("binaryFile")
    .load(f"/Volumes/{catalog}/pawshield/bootstrap/{source_glob}")
    .count()
)
row_count = spark.sql(
    f"SELECT COUNT(*) AS n FROM `{catalog}`.pawshield.email_extracts"
).collect()[0]["n"]
print(f"Source .eml files:           {file_count:>5}")
print(f"Rows in email_extracts:      {row_count:>5}")
if row_count > file_count:
    print(
        f"⚠  Row count exceeds file count by {row_count - file_count}.\n"
        f"   Likely cause: read_files() returned one-row-per-line\n"
        f"   instead of one-row-per-file. Confirm `wholetext => true`\n"
        f"   is on the read_files call in §3, then:\n"
        f"     DROP TABLE {catalog}.pawshield.email_extracts;\n"
        f"   and re-run §2 + §3."
    )
else:
    print("✓ Row count consistent with file count.")

Source .eml files:             500
Rows in email_extracts:        500
✓ Row count consistent with file count.


## 4. Verify — counts and Sarah's row

Quick verification:

* Total rows in `email_extracts` should match the number of `.eml`
  files (~500, per the setup-notebook generator output).
* `failOnError => false` rows with a null `extracted` are the
  ones that failed on the first pass — just re-run §3 to pick
  them up; the MERGE's WHERE filter targets exactly the null-
  extract rows so the retry costs only what the retry pays for.

In [0]:
%sql
SELECT
    COUNT(*)                                       AS total,
    SUM(CASE WHEN extracted IS NULL THEN 1 ELSE 0 END) AS null_extracts
FROM IDENTIFIER(:catalog || '.pawshield.email_extracts');

total,null_extracts
500,0


Running Sarah Chen's appeal email — the lifecycle anchor — through
the registered champion prompt produces a schema-valid JSON row,
but **schema-valid is not the same as value-accurate**: across a
batch, some fields drift — the model can under-classify the
`intent` of an appeal, or occasionally hallucinate an absent
`customer_id`. That is the deliberate few-shot lesson — few-shot
examples *shift the prior* but don't impose a hard constraint —
surfaced here at batch scale, not a defect to "fix" by re-tuning.

**Lifecycle continuity.** Sarah's case still routes correctly to
`approve_with_adjustment` via the ClaimClerk agent (c0901):
reasons independently over the policy text, not over this batch
`intent` label. Batch extraction feeds analytics/reporting, not
live routing — which is why extraction quality is *monitored*
(evaluation + monitoring), not assumed.

In [0]:
%sql
SELECT source_path, extracted, processed_at
FROM IDENTIFIER(:catalog || '.pawshield.email_extracts')
WHERE source_path LIKE '%sarah_chen_20260328_098473%';

source_path,extracted,processed_at
dbfs:/Volumes/genaicert/pawshield/bootstrap/claim_emails/sarah_chen_20260328_098473.eml,"List(CLM-2026-03-00471, CUST-Sarah-Chen-NNN, Biscuit, 512-555-0188, claim_status, frustrated, low)",2026-05-25T14:58:52.234Z


## What's next

* **The CI/CD workflow** uses the same pattern in a Databricks
  Workflow — `email_extracts` becomes a daily output the
  downstream analytics consume.
* **The ClaimClerk agent (c0901)** can use `email_extracts` as a
  batch-warmed cache: when the agent receives an email it's
  already seen, it skips the extraction step and reads the
  stored fields.
* **The evaluation notebook** can score the extraction quality by
  sampling rows from `email_extracts` and comparing to a
  hand-graded ground-truth subset.